### Phase 5: Machine Learning & Forecasting

#### Environment Setup

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Load the perfectly engineered dataset explicitly from the default database
df_features = spark.read.table("default.gold_model_ready_features")

# Filter out initial rows that have zero sales due to lack of historical lag data
df_features = df_features.filter((col("sales") > 0) & (col("lag_sales_7d") > 0))

print(f"✅ Successfully loaded Gold Features. Model-ready rows: {df_features.count():,}")

#### The Vector Assembler
As Spark Machine Learning algorithms cannot read 10 different columns at once. We require all to be bundled into a single mathematical array. For that it takes all your engineered columns (like lag_sales_7d, price_deviation_ratio, is_promoted) and crushes them together into one single master column called features.

In [0]:
# Define the exact features we engineered in Notebook 4
input_cols = [
    "lag_sales_1d", 
    "lag_sales_7d", 
    "rolling_avg_sales_7d",
    "is_promoted",
    "lag_promo_1d",
    "lag_promo_7d",
    "price_deviation_ratio",
    "hierarchy1_id_encoded", 
    "city_id_encoded", 
    "storetype_id_encoded",
    "cluster_id_encoded"
]

# Assemble them into a single mathematical vector called "features"
assembler = VectorAssembler(inputCols=input_cols, outputCol="features", handleInvalid="skip")
df_assembled = assembler.transform(df_features)

print("✅ Features successfully bundled into the target ML vector format.")

#### Chronological Train/Test Split
To test the model on the exact same data we used to train it,  we need to split the data: we teach the model using the past (80%), and we test its accuracy by having it predict the future (20%).

In [0]:
from pyspark.sql.functions import when

print("Executing Time-Series Data Split...")

# We use an 80/20 split to train the AI on the past to predict the future
train_df, test_df = df_assembled.randomSplit([0.7, 0.3], seed=42)

# Calculate the 99th percentile threshold from training data
outlier_threshold = train_df.approxQuantile("sales", [0.99], 0.01)[0]
print(f"📊 99th Percentile Sales Threshold: {outlier_threshold} units")

# Cap the extreme values in the training data only
train_df_capped = train_df.withColumn(
    "sales", 
    when(col("sales") > outlier_threshold, outlier_threshold).otherwise(col("sales"))
)

#### Model Training (Random Forest)
We need a system that can look at millions of past sales and figure out exactly how price cuts and promotions drive customer demand. For this we will use a Random Forest Regressor Machine Learning  model and feeds it the 80% training data. The model learns the patterns (e.g., When the price deviation is -15% and it was on promo last week, sales usually hit 4 units").

In [0]:
print("Initiating Machine Learning Model Training (Random Forest)...")

# Initialize the forecasting algorithm
rf = RandomForestRegressor(featuresCol="features", labelCol="sales", numTrees=60, maxDepth=12, seed=42)

# Train the model on the historical training data
rf_model = rf.fit(train_df)

print("✅ Model Training Complete! The model has successfully learned the promotional and pricing patterns.")

#### Model Evaluating Forecasting Accuracy
To check the model is ready to manage data. We do this by testing it on historical data and comparing its daily sales forecasts against actual sales. We validate it against past sales data using two benchmarks: RMSE (to catch major prediction failures) and MAE (to track everyday accuracy).

In [0]:
print("Evaluating Forecasting Accuracy on Future Test Set...")

# Force the trained model to predict sales on the unseen test data
predictions = rf_model.transform(test_df)

# Calculate RMSE (Measures overall prediction accuracy, heavily penalizes large misses)
evaluator_rmse = RegressionEvaluator(labelCol="sales", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

# Calculate MAE (Measures the average unit error per day)
evaluator_mae = RegressionEvaluator(labelCol="sales", predictionCol="prediction", metricName="mae")
mae = evaluator_mae.evaluate(predictions)

print("-" * 50)
print(f"📉 Root Mean Squared Error (RMSE): {rmse:.2f} units")
print(f"📉 Mean Absolute Error (MAE):      {mae:.2f} units per transaction")
print("-" * 50)
print(f"💡 Translation: On any given day, our model's forecast is only off by an average of {mae:.2f} units!")

#### Executive Visualization (Forecast vs. Actuals)
It takes a random sample of 150 days and uses Python model to draw a line chart comparing Actual Sales vs Predicted Sales.

In [0]:
print("Step 6: Generating Executive Forecast Verification Dashboard...")

# Pull a sample of actual vs predicted sales for visualization
sample_predictions = predictions.select("sales", "prediction").limit(150).toPandas()

plt.figure(figsize=(14, 6))
sns.set_theme(style="whitegrid")

# Plot Actual Sales (Blue) vs Predicted Sales (Orange)
plt.plot(sample_predictions.index, sample_predictions['sales'], label='Actual Sales (Reality)', color='#2980b9', linewidth=2, alpha=0.8)
plt.plot(sample_predictions.index, sample_predictions['prediction'], label='Model Forecast (Prediction)', color='#e67e22', linewidth=2, linestyle='--')

plt.title("Executive Validation: AI Forecast vs. Actual Revenue Drivers", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Sample Timeline (Store-Item Combinations)")
plt.ylabel("Units Sold")
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

#### Final Delivery & Dashboard Handoff

In [0]:
print("Committing Forecast Predictions to Executive Storage Matrix...")

# Keep only the readable business columns and the final prediction
final_output = predictions.select(
    "date", "store_id", "product_id", "price", "is_promoted", "sales", "prediction"
)

# Rename for extreme clarity
final_output = final_output.withColumnRenamed("sales", "actual_sales") \
                           .withColumnRenamed("prediction", "forecasted_demand")

# Save as a permanent Delta table for Executive Dashboards
final_output.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.gold_executive_forecast_predictions")

print("Saved final business output target: 'default.gold_executive_forecast_predictions'")